In [0]:
USE CATALOG smart_odoo;

-- =========================
-- CREATE SILVER TABLE
-- =========================

CREATE TABLE IF NOT EXISTS silver.sale_order
(
    sale_order_id       int,
    
    partner_id          int,
    partner_name        STRING,
    
    team_id             int,
    team_name           STRING,
    
    company_id          int,
    company_name        STRING,
    
    user_id             int,
    user_name           STRING,
    
    currency_id         int,
    currency_name       STRING,
    
    date_order          TIMESTAMP,
    order_state         STRING,
    amount_untaxed      DOUBLE,
    amount_tax          DOUBLE,
    amount_total        DOUBLE,
    pricelist_id        STRING,
    origin              STRING,
    client_order_ref    STRING,
    commitment_date     TIMESTAMP,
    validity_date       TIMESTAMP,
    created_at          TIMESTAMP,
    updated_at          TIMESTAMP,
    dwh_loaded_at       TIMESTAMP,
    dwh_source_db       STRING
);

-- =========================
-- MERGE BRONZE → SILVER
-- =========================

MERGE INTO silver.sale_order AS target
USING (
    SELECT
        cast( so.id  As int)                                              AS sale_order_id,

        -- Many2one fields: parse [id, name] JSON array
        cast( GET_JSON_OBJECT(so.partner_id,  '$[0]') As int)            AS partner_id,
        GET_JSON_OBJECT(so.partner_id,  '$[1]')                          AS partner_name,

        cast( GET_JSON_OBJECT(so.team_id,     '$[0]') As int)            AS team_id,
        GET_JSON_OBJECT(so.team_id,     '$[1]')                          AS team_name,

        cast( GET_JSON_OBJECT(so.company_id,  '$[0]') As int)            AS company_id,
        GET_JSON_OBJECT(so.company_id,  '$[1]')                          AS company_name,

        cast( GET_JSON_OBJECT(so.user_id,     '$[0]') As int)            AS user_id,
        GET_JSON_OBJECT(so.user_id,     '$[1]')                          AS user_name,

        cast( GET_JSON_OBJECT(so.currency_id, '$[0]') As int)            AS currency_id,
        GET_JSON_OBJECT(so.currency_id, '$[1]')                          AS currency_name,

        so.date_order                                      AS date_order,
        so.state                                           AS order_state,
        CAST(so.amount_untaxed  AS DOUBLE)                 AS amount_untaxed,
        CAST(so.amount_tax      AS DOUBLE)                 AS amount_tax,
        CAST(so.amount_total    AS DOUBLE)                 AS amount_total,
        so.pricelist_id,
        so.origin,
        so.client_order_ref,
        so.commitment_date        AS commitment_date,
        so.validity_date          AS validity_date,
        so.create_date            AS created_at,
        so.write_date             AS updated_at,
        current_timestamp()       AS dwh_loaded_at,
        so.dwh_source_db

    FROM bronze.sale_order so
    WHERE CAST(so.write_date AS TIMESTAMP) > COALESCE(
        (
            SELECT last_write_date
            FROM silver.load_tracking
            WHERE table_name = 'sale_order'
        ),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.sale_order_id = source.sale_order_id

WHEN MATCHED THEN UPDATE SET
    target.partner_id        = source.partner_id,
    target.partner_name      = source.partner_name,
    target.team_id           = source.team_id,
    target.team_name         = source.team_name,
    target.company_id        = source.company_id,
    target.company_name      = source.company_name,
    target.user_id           = source.user_id,
    target.user_name         = source.user_name,
    target.currency_id       = source.currency_id,
    target.currency_name     = source.currency_name,
    target.date_order        = source.date_order,
    target.order_state       = source.order_state,
    target.amount_untaxed    = source.amount_untaxed,
    target.amount_tax        = source.amount_tax,
    target.amount_total      = source.amount_total,
    target.pricelist_id      = source.pricelist_id,
    target.origin            = source.origin,
    target.client_order_ref  = source.client_order_ref,
    target.commitment_date   = source.commitment_date,
    target.validity_date     = source.validity_date,
    target.created_at        = source.created_at,
    target.updated_at        = source.updated_at,
    target.dwh_loaded_at     = current_timestamp(),
    target.dwh_source_db     = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT (
    sale_order_id, partner_id, partner_name, team_id, team_name,
    company_id, company_name, user_id, user_name, currency_id, currency_name,
    date_order, order_state, amount_untaxed, amount_tax, amount_total,
    pricelist_id, origin, client_order_ref, commitment_date, validity_date,
    created_at, updated_at, dwh_loaded_at, dwh_source_db
)
VALUES (
    source.sale_order_id, source.partner_id, source.partner_name,
    source.team_id, source.team_name, source.company_id, source.company_name,
    source.user_id, source.user_name, source.currency_id, source.currency_name,
    source.date_order, source.order_state, source.amount_untaxed,
    source.amount_tax, source.amount_total, source.pricelist_id,
    source.origin, source.client_order_ref, source.commitment_date,
    source.validity_date, source.created_at, source.updated_at,
    current_timestamp(), source.dwh_source_db
);




In [0]:
select * from smart_odoo.silver.sale_order

In [0]:
CREATE TABLE IF NOT EXISTS silver.sale_order_line
(
    sale_order_line_id          int,
    sale_order_id               int,
    sale_order_name             STRING,
    company_id                  int,
    company_name                STRING,
    currency_id                 int,
    currency_name               STRING,
    partner_id                  int,
    partner_name                STRING,
    salesman_id                 int,
    salesman_name               STRING,
    product_id                  int,
    product_name                STRING,
    product_uom_id              int,
    product_uom_name            STRING,
    order_state                 STRING,
    display_type                STRING,
    virtual_id                  int,
    linked_virtual_id           int,
    qty_delivered_method        STRING,
    invoice_status              STRING,
    analytic_distribution       STRING,
    extra_tax_data              STRING,
    line_name                   STRING,
    product_uom_qty             DOUBLE,
    price_unit                  DOUBLE,
    discount                    DOUBLE,
    price_subtotal              DOUBLE,
    price_total                 DOUBLE,
    price_reduce_taxexcl        DOUBLE,
    price_reduce_taxinc         DOUBLE,
    qty_delivered               DOUBLE,
    qty_invoiced                DOUBLE,
    qty_to_invoice              DOUBLE,
    untaxed_amount_invoiced     DOUBLE,
    untaxed_amount_to_invoice   DOUBLE,
    is_downpayment              STRING,
    is_expense                  STRING,
    collapse_prices             STRING,
    collapse_composition        STRING,
    created_at                  TIMESTAMP,
    updated_at                  TIMESTAMP,
    technical_price_unit        DOUBLE,
    price_tax                   DOUBLE,
    customer_lead               DOUBLE,
    is_optional                 STRING,
    dwh_loaded_at               TIMESTAMP,
    dwh_source_db               STRING
);

-- =========================
-- MERGE BRONZE → SILVER
-- =========================

MERGE INTO silver.sale_order_line AS target
USING (
    SELECT
        cast( sol.id As int)                                          AS sale_order_line_id,

        cast( GET_JSON_OBJECT(sol.order_id,           '$[0]') As int) AS sale_order_id,
        GET_JSON_OBJECT(sol.order_id,           '$[1]')               AS sale_order_name,

        cast( GET_JSON_OBJECT(sol.company_id,         '$[0]') As int) AS company_id,
        GET_JSON_OBJECT(sol.company_id,         '$[1]')               AS company_name,

        cast( GET_JSON_OBJECT(sol.currency_id,        '$[0]') As int)    AS currency_id,
        GET_JSON_OBJECT(sol.currency_id,        '$[1]')    AS currency_name,

        cast( GET_JSON_OBJECT(sol.order_partner_id,   '$[0]') As int)    AS partner_id,
        GET_JSON_OBJECT(sol.order_partner_id,   '$[1]')    AS partner_name,

        cast( GET_JSON_OBJECT(sol.salesman_id,        '$[0]') As int)    AS salesman_id,
        GET_JSON_OBJECT(sol.salesman_id,        '$[1]')    AS salesman_name,

        cast( GET_JSON_OBJECT(sol.product_id,         '$[0]') As int)    AS product_id,
        GET_JSON_OBJECT(sol.product_id,         '$[1]')    AS product_name,

        cast( GET_JSON_OBJECT(sol.product_uom_id,     '$[0]') As int)    AS product_uom_id,
        GET_JSON_OBJECT(sol.product_uom_id,     '$[1]')    AS product_uom_name,

        sol.state                                          AS order_state,
        sol.display_type,
        cast( sol.virtual_id As int),
        cast( sol.linked_virtual_id As int),
        sol.qty_delivered_method,
        sol.invoice_status,
        sol.analytic_distribution,
        sol.extra_tax_data,
        sol.name                                           AS line_name,

        CAST(sol.product_uom_qty            AS DOUBLE)     AS product_uom_qty,
        CAST(sol.price_unit                 AS DOUBLE)     AS price_unit,
        CAST(sol.discount                   AS DOUBLE)     AS discount,
        CAST(sol.price_subtotal             AS DOUBLE)     AS price_subtotal,
        CAST(sol.price_total                AS DOUBLE)     AS price_total,
        CAST(sol.price_reduce_taxexcl       AS DOUBLE)     AS price_reduce_taxexcl,
        CAST(sol.price_reduce_taxinc        AS DOUBLE)     AS price_reduce_taxinc,
        CAST(sol.qty_delivered              AS DOUBLE)     AS qty_delivered,
        CAST(sol.qty_invoiced               AS DOUBLE)     AS qty_invoiced,
        CAST(sol.qty_to_invoice             AS DOUBLE)     AS qty_to_invoice,
        CAST(sol.untaxed_amount_invoiced    AS DOUBLE)     AS untaxed_amount_invoiced,
        CAST(sol.untaxed_amount_to_invoice  AS DOUBLE)     AS untaxed_amount_to_invoice,
        sol.is_downpayment,
        sol.is_expense,
        sol.collapse_prices,
        sol.collapse_composition,
        CAST(sol.create_date                AS TIMESTAMP)  AS created_at,
        CAST(sol.write_date                 AS TIMESTAMP)  AS updated_at,
        CAST(sol.technical_price_unit       AS DOUBLE)     AS technical_price_unit,
        CAST(sol.price_tax                  AS DOUBLE)     AS price_tax,
        CAST(sol.customer_lead              AS DOUBLE)     AS customer_lead,
        sol.is_optional,
        current_timestamp()                                AS dwh_loaded_at,
        sol.dwh_source_db

    FROM bronze.sale_order_line sol
    WHERE CAST(sol.write_date AS TIMESTAMP) > COALESCE(
        (
            SELECT last_write_date
            FROM silver.load_tracking
            WHERE table_name = 'sale_order_line'
        ),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.sale_order_line_id = source.sale_order_line_id

WHEN MATCHED THEN UPDATE SET
    target.sale_order_id                = source.sale_order_id,
    target.sale_order_name              = source.sale_order_name,
    target.company_id                   = source.company_id,
    target.company_name                 = source.company_name,
    target.currency_id                  = source.currency_id,
    target.currency_name                = source.currency_name,
    target.partner_id                   = source.partner_id,
    target.partner_name                 = source.partner_name,
    target.salesman_id                  = source.salesman_id,
    target.salesman_name                = source.salesman_name,
    target.product_id                   = source.product_id,
    target.product_name                 = source.product_name,
    target.product_uom_id               = source.product_uom_id,
    target.product_uom_name             = source.product_uom_name,
    target.order_state                  = source.order_state,
    target.display_type                 = source.display_type,
    target.virtual_id                   = source.virtual_id,
    target.linked_virtual_id            = source.linked_virtual_id,
    target.qty_delivered_method         = source.qty_delivered_method,
    target.invoice_status               = source.invoice_status,
    target.analytic_distribution        = source.analytic_distribution,
    target.extra_tax_data               = source.extra_tax_data,
    target.line_name                    = source.line_name,
    target.product_uom_qty              = source.product_uom_qty,
    target.price_unit                   = source.price_unit,
    target.discount                     = source.discount,
    target.price_subtotal               = source.price_subtotal,
    target.price_total                  = source.price_total,
    target.price_reduce_taxexcl         = source.price_reduce_taxexcl,
    target.price_reduce_taxinc          = source.price_reduce_taxinc,
    target.qty_delivered                = source.qty_delivered,
    target.qty_invoiced                 = source.qty_invoiced,
    target.qty_to_invoice               = source.qty_to_invoice,
    target.untaxed_amount_invoiced      = source.untaxed_amount_invoiced,
    target.untaxed_amount_to_invoice    = source.untaxed_amount_to_invoice,
    target.is_downpayment               = source.is_downpayment,
    target.is_expense                   = source.is_expense,
    target.collapse_prices              = source.collapse_prices,
    target.collapse_composition         = source.collapse_composition,
    target.created_at                   = source.created_at,
    target.updated_at                   = source.updated_at,
    target.technical_price_unit         = source.technical_price_unit,
    target.price_tax                    = source.price_tax,
    target.customer_lead                = source.customer_lead,
    target.is_optional                  = source.is_optional,
    target.dwh_loaded_at                = current_timestamp(),
    target.dwh_source_db                = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT (
    sale_order_line_id, sale_order_id, sale_order_name,
    company_id, company_name, currency_id, currency_name,
    partner_id, partner_name, salesman_id, salesman_name,
    product_id, product_name, product_uom_id, product_uom_name,
    order_state, display_type, virtual_id, linked_virtual_id,
    qty_delivered_method, invoice_status, analytic_distribution,
    extra_tax_data, line_name, product_uom_qty, price_unit,
    discount, price_subtotal, price_total, price_reduce_taxexcl,
    price_reduce_taxinc, qty_delivered, qty_invoiced, qty_to_invoice,
    untaxed_amount_invoiced, untaxed_amount_to_invoice,
    is_downpayment, is_expense, collapse_prices, collapse_composition,
    created_at, updated_at, technical_price_unit, price_tax,
    customer_lead, is_optional, dwh_loaded_at, dwh_source_db
)
VALUES (
    source.sale_order_line_id, source.sale_order_id, source.sale_order_name,
    source.company_id, source.company_name, source.currency_id, source.currency_name,
    source.partner_id, source.partner_name, source.salesman_id, source.salesman_name,
    source.product_id, source.product_name, source.product_uom_id, source.product_uom_name,
    source.order_state, source.display_type, source.virtual_id, source.linked_virtual_id,
    source.qty_delivered_method, source.invoice_status, source.analytic_distribution,
    source.extra_tax_data, source.line_name, source.product_uom_qty, source.price_unit,
    source.discount, source.price_subtotal, source.price_total, source.price_reduce_taxexcl,
    source.price_reduce_taxinc, source.qty_delivered, source.qty_invoiced, source.qty_to_invoice,
    source.untaxed_amount_invoiced, source.untaxed_amount_to_invoice,
    source.is_downpayment, source.is_expense, source.collapse_prices, source.collapse_composition,
    source.created_at, source.updated_at, source.technical_price_unit, source.price_tax,
    source.customer_lead, source.is_optional, current_timestamp(), source.dwh_source_db
);

-- =========================
-- UPDATE TRACKING
-- =========================

MERGE INTO silver.load_tracking AS t
USING (
    SELECT
        'sale_order_line'                           AS table_name,
        MAX(CAST(write_date AS TIMESTAMP))          AS last_write_date
    FROM bronze.sale_order_line
    WHERE CAST(write_date AS TIMESTAMP) > COALESCE(
        (
            SELECT last_write_date
            FROM silver.load_tracking
            WHERE table_name = 'sale_order_line'
        ),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS s
ON t.table_name = s.table_name
WHEN MATCHED AND s.last_write_date IS NOT NULL
    THEN UPDATE SET t.last_write_date = s.last_write_date
WHEN NOT MATCHED
    THEN INSERT (table_name, last_write_date)
    VALUES ('sale_order_line', CAST('1900-01-01' AS TIMESTAMP));